# CPCV 6,2 Debug — SPY 2024 dev set
Objetivo: verificar que el CPCV está bien implementado antes de usarlo en producción.

- **Dev set**: 2024-01-01 → 2024-12-31 (8 meses train / 4 meses test por fold rolling)
- **N=6 grupos, k=2 test** → C(6,2)=15 splits, φ=5 paths
- Las gráficas se guardan en `resultados/debug/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from cpcv_analysis.config import (
    XGB_PARAMS, WF_START, DEV_START, DEV_END,
    RETRAIN_START, RETRAIN_END, HOLDOUT_START, HOLDOUT_END, START, END,
)
from cpcv_analysis.data import load_asset
from cpcv_analysis.backtest_engine import (
    slice_by_dates,
    _build_cpcv_splits_table, cpcv_sharpe_dist,
    cpcv_debug,
)
from cpcv_analysis.cv_runner import get_paths
from cpcv_analysis.plots import (
    plot_temporal_partition,
    plot_split_timelines,
    plot_is_oos_sharpe_per_split,
    plot_path_sharpes_dist,
)

OUT = 'resultados/debug/'

clf = XGBClassifier(**XGB_PARAMS)

# ── Cargar datos (usa cache CSV en data_cache/) ───────────────────────────────
X_full, y_full, t1_full, prices, fwd_ret_full = load_asset('SPY', START, END)
print(f'Datos completos: {len(X_full)} obs  {X_full.index[0].date()} → {X_full.index[-1].date()}')

# ── Slice dev set ─────────────────────────────────────────────────────────────
X_dev, y_dev, t1_dev, fwd_ret_dev = slice_by_dates(
    X_full, y_full, t1_full, fwd_ret_full, DEV_START, DEV_END)

print(f'Dev set:  {len(X_dev)} obs  {X_dev.index[0].date()} → {X_dev.index[-1].date()}')

## Figura 0 — Partición temporal

In [ ]:
plot_temporal_partition(
    prices,
    wf_start=WF_START,
    dev_start=DEV_START,
    dev_end=DEV_END,
    retrain_start=RETRAIN_START,
    retrain_end=RETRAIN_END,
    holdout_start=HOLDOUT_START,
    holdout_end=HOLDOUT_END,
    out_dir=OUT,
)
from IPython.display import Image
Image(OUT + '00_temporal_partition.png')

## Sección 1 — Tabla de splits + timeline visual

In [ ]:
splits_info, oos_by_split, is_by_split, preds_by_split = _build_cpcv_splits_table(
    clf, X_dev, y_dev, t1_dev, fwd_ret_dev)

print(f'Total splits: {len(splits_info)}')
rows = []
for s in splits_info:
    rows.append({
        'split_id':    s['split_id'],
        'test_groups': str(s['test_groups']),
        'n_train':     s['n_train'],
        'n_test':      s['n_test'],
        'n_purged':    s['n_purged'],
        'n_embargoed': s['n_embargoed'],
        'test_start':  oos_by_split[s['split_id']].index[0].date(),
        'test_end':    oos_by_split[s['split_id']].index[-1].date(),
    })
from IPython.display import display
display(pd.DataFrame(rows).set_index('split_id'))

In [ ]:
plot_split_timelines(splits_info, n_obs=len(X_dev), out_dir=OUT)
Image(OUT + '01_split_timelines.png')

## Sección 2 — IS vs OOS Sharpe por split
Sharpe = √252 · mean(pnl) / std(pnl) calculado sobre el PnL concatenado de cada split.

In [ ]:
def _sr(pnl):
    if len(pnl) < 2 or pnl.std() == 0:
        return 0.0
    return float(np.sqrt(252) * pnl.mean() / pnl.std())

print('Split  test_groups   IS_SR    OOS_SR   n_train  n_test')
print('-' * 60)
for s in splits_info:
    sid = s['split_id']
    is_sr  = _sr(is_by_split[sid])
    oos_sr = _sr(oos_by_split[sid])
    print(f"  {sid:>2}   {str(s['test_groups']):<12}  {is_sr:+.3f}   {oos_sr:+.3f}   "
          f"{s['n_train']:>5}   {s['n_test']:>5}")
    print(f"         IS:  √252 · {is_by_split[sid].mean():.6f} / {is_by_split[sid].std():.6f} = {is_sr:+.4f}")
    print(f"         OOS: √252 · {oos_by_split[sid].mean():.6f} / {oos_by_split[sid].std():.6f} = {oos_sr:+.4f}")

In [ ]:
plot_is_oos_sharpe_per_split(splits_info, is_by_split, oos_by_split, out_dir=OUT)
Image(OUT + '02_is_oos_sharpe_per_split.png')

## Sección 3 — Paths: construcción y Sharpes
Cada path = N/k splits disjuntos que cubren todos los grupos exactamente una vez.
SR del path = √252 · mean(pnl_concatenado_del_path) / std(pnl_concatenado_del_path)

In [ ]:
paths = get_paths(6, 2)  # N=6, k=2 → φ=5 paths, 3 splits/path
print(f'φ = {len(paths)} paths, {len(paths[0])} splits/path')
print()

path_sharpes_debug = []
for pid, split_ids in enumerate(paths):
    valid = [sid for sid in split_ids if sid in oos_by_split]
    path_pnl = pd.concat([oos_by_split[sid] for sid in valid]).sort_index()
    sr = _sr(path_pnl)
    path_sharpes_debug.append(sr)
    print(f'Path {pid}: splits={split_ids}  n_obs={len(path_pnl)}')
    print(f'  SR = √252 · {path_pnl.mean():.6f} / {path_pnl.std():.6f} = {sr:+.4f}')
    print(f'  Rango: {path_pnl.index[0].date()} → {path_pnl.index[-1].date()}')

path_sharpes_debug = pd.Series(path_sharpes_debug, name='cpcv_debug')
print(f'\nSharpes debug: {path_sharpes_debug.values}')

## Sección 4 — cpcv_prod: verificar coincidencia

In [ ]:
cpcv_prod = cpcv_sharpe_dist(clf, X_dev, y_dev, t1_dev, fwd_ret_dev)
print('Sharpes CPCV prod:', cpcv_prod.values)
print('Sharpes CPCV debug:', path_sharpes_debug.values)

coinciden = np.allclose(np.sort(path_sharpes_debug.values), np.sort(cpcv_prod.values), atol=1e-6)
print(f'¿Coinciden debug == prod? {coinciden}')

In [ ]:
plot_path_sharpes_dist(path_sharpes_debug, cpcv_prod_sharpes=cpcv_prod, out_dir=OUT)
Image(OUT + '03_path_sharpes_dist.png')

## Sección 5 — cpcv_debug completo (verbose)
Muestra tablas de splits, series temporales por split, fórmulas IS/OOS verificables, returns del path 0 y distribución de Sharpes.

In [ ]:
cpcv_sharpes_verbose = cpcv_debug(clf, X_dev, y_dev, t1_dev, fwd_ret_dev)
print('\nSharpes de paths (cpcv_debug):')
print(cpcv_sharpes_verbose.to_string())